# Instantiate IPOPT porblem to solve with and without track angles objective

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname("collocation_outputs.ipynb"), ".."))
from pathlib import Path

import yaml, tempfile, os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr  
import matplotlib.pyplot as plt

from biosym.visualization.stickfigure import plot_stick_figure
from biosym.ocp import collocation
from biosym.ocp.utils import *

In [ ]:
prob_stand = collocation.Collocation("tests/collocation/standing2d.yaml", force_rebuild=False)
(x, globals_dict), info = prob_stand.solve(visualize=True) 

In [ ]:
prob = collocation.Collocation("tests/collocation/walking2d.yaml", force_rebuild=False)
(x, globals_dict), info = prob.solve(visualize=False)

In [ ]:
ani = plot_stick_figure(prob.model, 
                  (x, None))

In [ ]:
from IPython.display import HTML, display

display(HTML(ani.to_jshtml()))

In [ ]:
plt.plot(prob.objective._objectives[-2].obj_settings['q_exp'])
#add prob.model.state_vector as the legend
plt.title('Expected joint angles')
plt.xlabel('Time step')
plt.ylabel('Angle (rad)')
plt.legend(prob.model.state_vector[3:9])
prob.objective._objectives[-2].obj_settings['q_exp'].shape


In [ ]:
plt.plot(prob.objective._objectives[-1].obj_settings['grf_exp'])
#add prob.model.state_vector as the legend
plt.title('Expected GRFs')
plt.xlabel('Time step')
plt.legend(prob.model.state_vector[33:45])
prob.objective._objectives[-1].obj_settings['grf_exp'].shape

# Get solution outputs

In [ ]:
x.states.model.shape # (n_nodes, n_states)

In [ ]:
prob.model.state_vector # all states

In [ ]:
coords = prob.model.ext_forces   # dict with 'idx' and 'n'
print(coords)

In [ ]:
name = "q_hip_r"
idx = prob.model.state_vector.index(name) 
print(idx)

In [ ]:
coords = prob.model.coordinates   # dict with 'idx' and 'n'
start, n_coords = coords['idx'], coords['n']

q_sim = x.states.model[:, start : start + n_coords]  # shape (n_nodes, n_coords)
print("q_sim shape:", q_sim.shape)

# Plot joint angles with and without track angles obective

In [ ]:
qexp_path = Path("~/.biosym/Data/Subject_01/gait_avg_joint_angles.csv").expanduser()
q_exp = pd.read_csv(qexp_path)
q_exp_knee_r = q_exp["knee_angle_r_mean"].to_numpy()
q_exp_knee_l = q_exp["knee_angle_l_mean"].to_numpy()

# simple single plot: two simulated (solid) and two experimental (dotted)
fig, ax = plt.subplots(figsize=(8, 4))

names = ("q_knee_l", "q_knee_r")
exp = {"q_knee_l": q_exp_knee_l, "q_knee_r": q_exp_knee_r}

for name in names:
    idx = prob.model.state_vector.index(name)
    print(idx)
    sim_line, = ax.plot(np.asarray(x.states.model[:, idx]), label=name)
    color = sim_line.get_color()
    ax.plot(np.linspace(0, len(x.states.model) - 1, len(exp[name])), exp[name], linestyle=':', color=color, label=f"{name} (exp)")

ax.legend()
ax.set_title("Knee angles with tracking angles objective")
plt.tight_layout()
plt.show()

In [ ]:
prob.model.state_vector

In [ ]:
# simple single plot: two simulated (solid) and two experimental (dotted)
fig, ax = plt.subplots(figsize=(8, 4))

names = ["f_foot_r_y"]
# load the experimental GRF CSV (adjust path if already loaded above)
grfexp_path = Path("~/.biosym/Data/Subject_01/gait_avg_grfs.csv").expanduser()
grf_exp = pd.read_csv(grfexp_path)

# choose the experimental column that corresponds to the simulated state
# change this to the exact column name from your CSV if different
exp_col = "ground_force_vy_mean"

for name in names:
    sim = np.asarray(x.states.model[:, idx])
    sim_line, = ax.plot(sim, label=name)
    color = sim_line.get_color()

    if exp_col in grf_exp.columns:
        exp_series = grf_exp[exp_col].to_numpy()
        # align lengths: map experimental samples onto simulation sample indices
        t_sim = np.arange(len(sim))
        t_exp = np.linspace(0, len(sim) - 1, len(exp_series))
        ax.plot(t_exp, exp_series, linestyle=":", color=color, label=f"{name} (exp)")
    else:
        print(f"exp column '{exp_col}' not found in {grfexp_path}; skipping exp plot")

ax.legend()
ax.set_title("GRF Comparison")
plt.tight_layout()
plt.show()

In [ ]:
grfexp_path = Path("~/.biosym/Data/Subject_01/gait_avg_grfs.csv").expanduser()
grf_exp = pd.read_csv(grfexp_path)
grf_exp_knee_r = grf_exp["knee_angle_r_mean"].to_numpy()
grf_exp_knee_l = grf_exp["knee_angle_l_mean"].to_numpy()

# simple single plot: two simulated (solid) and two experimental (dotted)
fig, ax = plt.subplots(figsize=(8, 4))

names = ("grf_knee_l", "grf_knee_r")
exp = {"f_knee_l": grf_exp_knee_l, "grf_knee_r": grf_exp_knee_r}

for name in names:
    idx = prob.model.state_vector.index(name)
    print(idx)
    sim_line, = ax.plot(np.asarray(x.states.model[:, idx]), label=name)
    color = sim_line.get_color()
    ax.plot(np.linspace(0, len(x.states.model) - 1, len(exp[name])), exp[name], linestyle=':', color=color, label=f"{name} (exp)")

ax.legend()
ax.set_title("Knee angles with tracking angles objective")
plt.tight_layout()
plt.show()

# Metrics

In [ ]:
q_exp = pd.read_csv("~/.biosym/Data/Subject_01/gait_avg_joint_angles.csv").filter(like="mean")
coords = prob.model.coordinates   # dict with 'idx' and 'n'
start, n_coords = coords['idx'], coords['n']

q_sim = x.states.model[:, start : start + n_coords]  # shape (n_nodes, n_coords)

In [ ]:
def compare_joint_angles(a, b):
    """a,b: arrays shape (N, D). Returns dict of lists (length D)."""
    a = np.asarray(a)
    b = np.asarray(b)
    assert a.shape == b.shape
    N, D = a.shape
    res = {
        "pearson": np.zeros(D),
        "spearman": np.zeros(D),
        "cosine": np.zeros(D),
        "rmse": np.zeros(D),
        "mae": np.zeros(D),
    }
    for j in range(D):
        x = a[:, j]
        y = b[:, j]
        # Pearson (numpy)
        c = np.corrcoef(x, y)[0, 1] if x.std() != 0 and y.std() != 0 else np.nan
        res["pearson"][j] = c
        # Spearman (fallback to nan if scipy missing)
        if spearmanr is not None:
            res["spearman"][j] = spearmanr(x, y).correlation
        else:
            res["spearman"][j] = np.nan
        # Cosine
        denom = (np.linalg.norm(x) * np.linalg.norm(y))
        res["cosine"][j] = (np.dot(x, y) / denom) if denom != 0 else np.nan
        # Errors
        err = x - y
        rmse = np.sqrt(np.mean(err ** 2))
        res["rmse"][j] = rmse
        res["mae"][j] = np.mean(np.abs(err))

    return res

In [ ]:
joint_angle_metrics = compare_joint_angles(q_exp, q_sim[:-1])

In [ ]:
keys = [k for k in joint_angle_metrics.keys()]
D = joint_angle_metrics[keys[0]].shape[0]
x = np.arange(D)

# try to get labels from q_exp if it's a DataFrame
labels = [c.replace('_mean', '') for c in q_exp.columns]

colors = plt.get_cmap("tab10").colors
for key in keys:
    a = np.asarray(joint_angle_metrics[key])
    fig, ax = plt.subplots(figsize=(8, 4))
    # series lines for overview
    ax.plot(x, a, '-k', lw=1, label='with track_angles', marker='o')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(key)
    ax.legend()
    plt.tight_layout()
    plt.show()